Runs Great Expectations validation against the current `weather_curated` table.
Produces a structured pass/fail report and an HTML data-docs summary.
Runs independently after the main ingestion notebook, reading committed data
rather than sharing session state — avoids the `restartPython()` disruption
that installing Great Expectations requires.

In [0]:
%pip install great_expectations
dbutils.library.restartPython()

### Setup
Reload shared config after the Python session restart above.

In [0]:
%run ./00_setup_config

### Load current curated data
Validates the full table as it stands right now. this notebook checks the
*state* of the data, not just the latest run's new rows.

In [0]:
df_valid = spark.table("internship_databricks_ws.default.weather_curated")
print(f"Rows to validate: {df_valid.count()}")

%md
### Run Great Expectations validation suite

In [0]:
import great_expectations as gx

context = gx.get_context()
data_source = context.data_sources.add_spark(name="databricks_spark")
data_asset = data_source.add_dataframe_asset(name="weather_curated_asset")
batch_definition = data_asset.add_batch_definition_whole_dataframe("weather_batch")
batch = batch_definition.get_batch(batch_parameters={"dataframe": df_valid})

suite = gx.ExpectationSuite(name="weather_quality_suite")
suite.add_expectation(gx.expectations.ExpectColumnValuesToNotBeNull(column="city"))
suite.add_expectation(gx.expectations.ExpectColumnValuesToNotBeNull(column="weather_time_pkt"))
suite.add_expectation(gx.expectations.ExpectColumnValuesToBeBetween(column="temperature_c", min_value=-30, max_value=55))
suite.add_expectation(gx.expectations.ExpectColumnValuesToBeBetween(column="humidity_pct", min_value=0, max_value=100))
suite.add_expectation(gx.expectations.ExpectColumnValuesToBeBetween(column="windspeed_kmh", min_value=0, max_value=200))
suite.add_expectation(gx.expectations.ExpectColumnValuesToNotBeNull(column="latitude"))
suite.add_expectation(gx.expectations.ExpectColumnValuesToNotBeNull(column="longitude"))

results = batch.validate(suite)
print(f"Great Expectations validation success: {results.success}")

%md
### Report failures, if any

In [0]:
failure_count = 0
for r in results.results:
    if not r.success:
        failure_count += 1
        print(f"FAILED: {r.expectation_config.type} — {r.result}")

if failure_count == 0:
    print("All expectations passed.")
else:
    print(f"{failure_count} expectation(s) failed — review above.")

%md
### Log the result as a queryable record
Writes a one-row summary (timestamp, success flag, failure count) to a
`weather_dq_log` table — builds a history of data quality over time, so you
can show a trend, not just a single point-in-time check.

In [0]:
from pyspark.sql import Row
from datetime import datetime

dq_log_row = spark.createDataFrame([Row(
    check_time=datetime.utcnow(),
    success=results.success,
    rows_checked=df_valid.count(),
    failure_count=failure_count
)])

spark.sql(f"""
CREATE TABLE IF NOT EXISTS internship_databricks_ws.default.weather_dq_log
USING DELTA
LOCATION '{gold_path}weather_dq_log/'
""")

dq_log_row.write.format("delta").mode("append").option("mergeSchema", "true").saveAsTable("internship_databricks_ws.default.weather_dq_log")
print("Logged this validation run to weather_dq_log")

In [0]:
if not results.success:
    raise Exception(f"Data quality validation failed: {failure_count} expectation(s) did not pass. See weather_dq_log for details.")
else:
    print("Data quality check passed — safe to proceed to gold layer.")